In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/RRCA_baseflow.csv')

### Reformatting the dates from days since 01/01/0000 to M-D-Y
--------------------------------------------------------------

In [7]:
baseflow_df = df.copy()
# offset is the number of days between 01/01/0000 and 01/01/1900 (693963)
offset = 693963
baseflow_df['days_since_1900'] = baseflow_df.Date - offset
baseflow_df['date'] = pd.to_datetime(baseflow_df.days_since_1900.astype('int'), unit='D', origin=pd.Timestamp('1900-01-01'))
print(baseflow_df['date'])

0       1945-11-30
1       1945-12-31
2       1946-01-28
3       1946-02-28
4       1946-03-30
           ...    
15586   1976-03-30
15587   1976-04-30
15588   1976-05-30
15589   1976-06-30
15590   1976-07-31
Name: date, Length: 15591, dtype: datetime64[us]


### Preliminary Column Analysis (so we know what we're working with)
--------------------------------------------------------------------

**Where is our data from?**

Notes:
- The dataset is labeled *RRCA_baseflow.csv*
    - In the context of bodies of water, *RRCA* might refer to *Republican River Compact Administration*.
        - The *Republican River* is a river that runs east from Colorado through Nebraska and Kansas.
- It looks like the RRCA made a groundwater flow model to represent characteristics of the Republican River Basin.

Website: (https://www.republicanrivercompact.org/)

In [32]:
print(f"Minimum x coordinate(...?): {min(baseflow_df['x'])}")
print(f"Maximum x coordinate: {max(baseflow_df['x'])}")
print(f"Minimum y coordinate: {min(baseflow_df['y'])}")
print(f"Maximum y coordinate: {max(baseflow_df['y'])}")

Minimum x coordinate(...?): 721870
Maximum x coordinate: 1941550
Minimum y coordinate: 14387040
Maximum y coordinate: 14767200


I'm not sure what to make off these positions/coordinates. They don't seem to be WGS84 coordinates.

In [ ]:
print(min(baseflow_df['date']))
print(max(baseflow_df['date']))

1939-09-30 00:00:00
2000-08-30 00:00:00


The dataset spans 1939-2000. This doesn't line up with the initial scope of the model (1918-2000).

Here's the documentation on the model: (https://www.republicanrivercompact.org/v12p/RRCAModelDocumentation.pdf)
After digging in the documentation, I found that the model works on the premise that there was no recharge from ground water irrigation prior to 1940 (pg. 14). (I'm sorry if that understanding misrepresents the paper)

In [ ]:
pre40s = baseflow_df[baseflow_df.date < '1940-01-01']
print(pre40s[pre40s.Irrigation_pumping != 0])

Empty DataFrame
Columns: [Date, Segment_id, x, y, Evapotranspiration, Precipitation, Irrigation_pumping, Observed, days_since_1900, date]
Index: []
3183      22.278689
3184      34.028689
3185      53.620968
3186      81.770968
3303      13.793443
3304      14.593443
3305      15.822581
3306      17.522581
3311      70.885246
3312      72.685246
3313      73.285484
3314      72.635484
6319      27.272131
6320      64.222131
6321      92.725806
6322     112.825806
6391      38.360656
6392     116.360656
6393     169.064520
6394     196.064520
6510      13.524590
6511      41.024590
6512      65.403230
6513      86.903230
6629      10.081967
6630      30.581967
6631      57.209677
6632      90.709677
8377       4.147316
8378       2.848046
8379       1.972981
8380       1.529100
9109       2.574906
9110       7.810549
9111      10.115329
9112       9.379687
9815      28.862295
9816      36.512295
9817      44.367742
9818      52.567742
10013      8.190164
10014     12.440164
10015     15

So, the dataset is from the inaugural/2003 use of the model (also referred to as Version 12p). We can use this to explain the dataset and interpret certain variables (notably what the coordinates mean).